# 02 - SCiO device health

Connect to a Consumer Physics SCiO over USB and read everything it will tell you
about itself: identifiers, battery, temperature, and the read-only headers of the
firmware and calibration-table files it holds.

**This notebook does not capture scans** - use
[`01_scio_scan_to_spectrum.ipynb`](01_scio_scan_to_spectrum.ipynb) for that. Keeping
capture in one place means there is one scan format and one calibration policy,
rather than two that can drift apart.

Use this notebook when you want to check the device is healthy and awake, or to
record its firmware file headers. Those checksums are what you would match a
recovered firmware blob against - see
[`dev/notebooks/02_scio_keyrecovery.ipynb`](dev/notebooks/02_scio_keyrecovery.ipynb).

**Safety.** Only read-only commands are sent. Write and state-changing commands
(parameter set, LED, file download, reset, rename) are never issued
automatically; the transport refuses them unless you explicitly pass
`allow_write=True`.

**Before you start:** turn the SCiO on (long press until steady blue) and plug in
USB. A slow light/dark pulse means it is idle or charging, and it will not answer.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from scio import protocol, store, usb

## 1. Find and connect

In [ ]:
ports = usb.find_scio_ports()
for p in ports:
    print(("* " if p["is_scio"] else "  ") + f'{p["device"]:8s} {p["vidpid"]}  {p["description"]}')
scio_port = next((p["device"] for p in ports if p["is_scio"]), None)
assert scio_port, "SCiO (VID:PID 0451:16AA) not found. Turn it on with a long press and replug USB."
print("Using", scio_port)

In [ ]:
dev = usb.ScioUSB(scio_port).open()
info = dev.read_device_info()
for k, v in info.items():
    print(f"  {k:18s} {v}")
if info.get("i2s_tag_missing"):
    print("\n  !! i2s_tag_config is empty - the server rejects scans without it.")
    print("     Power-cycle the device and re-run this cell before capturing anything.")

## 2. Battery and temperature

`cmos_t` is the exact Aptina value; `cmos_t_app` is the same reading after the
app's double truncation, and it is the one the calibration rules compare. See
README section 3 (`0x04`).

In [ ]:
battery = dev.read_battery()
temp = dev.read_temperature()
print("battery:    ", battery)
print("temperature:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in temp.items()})

## 3. Read-only firmware file headers

Each header is `(type, size, version, checksum)` as four `u32` LE. These identify
which per-device image-to-spectrum tables and DSP firmware the SCiO holds, and the
size + checksum are what you would validate a recovered `dsp_op` against.

Only headers can be read back - **bodies cannot**. See README section 5.

In [ ]:
headers = dev.read_all_file_headers()   # default ids: 87, 89-92, 99, 100-103
file_list = dev.read_file_list()
record = {"device": info, "file_list": file_list,
          "file_headers": {str(k): v for k, v in headers.items()}}
dev_path = store.save_device_files(record)

print(f'{"name":20s} {"id":>4s} {"size":>8s} {"ver":>5s} {"checksum":>12s}')
for fid, h in headers.items():
    name = protocol.FIRMWARE_FILE_NAMES.get(fid, str(fid))
    w = h.get("words", [])
    size, ver = (w[1], w[2]) if len(w) >= 3 else ("-", "-")
    print(f'{name:20s} {fid:4d} {size!s:>8s} {ver!s:>5s} {h.get("checksum")!s:>12s}')
print("\nsaved:", dev_path)

In [ ]:
dev.close()
print("Disconnected.")

## Where to go next

- **Capture a scan:** [`01_scio_scan_to_spectrum.ipynb`](01_scio_scan_to_spectrum.ipynb)
- **Probe undocumented opcodes:** [`03_scio_probe.ipynb`](03_scio_probe.ipynb)
- **Work on offline decoding:** [`dev/notebooks/02_scio_keyrecovery.ipynb`](dev/notebooks/02_scio_keyrecovery.ipynb)
  and [`dev/README.md`](dev/README.md), which explains why it is unsolved.